In [1]:
import json
import torch
from sentence_transformers import SentenceTransformer, util

In [2]:
with open('../data/exemples.json', encoding='utf-8') as file:
    exemples = json.load(file)

In [7]:
inputs = [item["input"] for item in exemples]
outputs = [item["output"] for item in exemples]
print('-'*15)
for i, p in enumerate(inputs):
    print(f"{i} - {p}")

---------------
0 - John is eating a ham sandwich at McDonald's.
1 - A man is playing an instrument.
2 - Simon didn't call me back. He must be busy.
3 - No politicians attended the meeting.
4 - He has the french nationality.
5 - The chef cooked dinner for the guests last night.
6 - The game took place in New York during christmas.
7 - While driving to work, he saw a deer.
8 - His dog is allergic to chocolate.
9 - He seems angry, he might leave the room.
10 - The human body contains two kidneys.
11 - Three birds are sitting on the roof.
12 - He studied all night in order to pass the exam.
13 - Smoking often leads to cancer.
14 - He uses his phone as a microphone.
15 - He must be sick.
16 - His boss told him to stay at work.
17 - Five cats are fighting.


Lors du classement de similarité, on veut classer en fonction des relations plutot que le sens de la phrase. Si deux phrases traitent du même sujet cela n'est pas pertinent, à moins que des relations similaires soit utilisées.

Pour : "His phone has two batteries." on veut que les phrases [0, 10] soient parmis les premières phrases dans le classement car elles ont des relations de contenance.  

## Sentence-BERT

In [4]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
exemples_embeddings = model.encode(inputs)

sentence = "Luc is working on a project with his interns, Younes and Ava. They all work at LISN."
print(sentence)
sentence_embeddings = model.encode(sentence)

Luc is working on a project with his interns, Younes and Ava. They all work at LISN.


In [9]:
scores = util.cos_sim(sentence_embeddings, exemples_embeddings)[0]
print(sentence)
for i in torch.argsort(scores, descending=True):
    print(f"score : {scores[i]:.4f} | Phrase : {inputs[i]}")

Luc is working on a project with his interns, Younes and Ava. They all work at LISN.
score : 0.2809 | Phrase : He has the french nationality.
score : 0.1678 | Phrase : He uses his phone as a microphone.
score : 0.1487 | Phrase : His boss told him to stay at work.
score : 0.1274 | Phrase : He seems angry, he might leave the room.
score : 0.1166 | Phrase : Simon didn't call me back. He must be busy.
score : 0.1037 | Phrase : He studied all night in order to pass the exam.
score : 0.0961 | Phrase : Three birds are sitting on the roof.
score : 0.0745 | Phrase : He must be sick.
score : 0.0598 | Phrase : His dog is allergic to chocolate.
score : 0.0542 | Phrase : The chef cooked dinner for the guests last night.
score : 0.0408 | Phrase : The human body contains two kidneys.
score : 0.0352 | Phrase : The game took place in New York during christmas.
score : 0.0259 | Phrase : No politicians attended the meeting.
score : 0.0081 | Phrase : A man is playing an instrument.
score : 0.0004 | Phrase

## Parser syntaxique